# Quintile Calculation and Export

This notebook reads `data/final_dataset.csv`, calculates the quintiles for key variables (Median House Price, Total Crime, Violence Against The Person, Total IMD), cleans up the PTAL categories for Transport, and exports the resulting dataset to `quintiles.csv`.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Read the dataset
df = pd.read_csv('data/final_dataset.csv')
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")

In [3]:
# 2. Calculate quintiles with string labels (e.g. 'Q1 (0 to 10)')
def calculate_quintile_with_labels(series, decimals=0):
    quantiles = series.quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0]).values
    labels = []
    format_str = f"{{:.{decimals}f}} to {{:.{decimals}f}}"
    for i in range(5):
        labels.append(f"Q{i+1} ({format_str.format(quantiles[i], quantiles[i+1])})")
    try:
        return pd.qcut(series, q=5, labels=labels)
    except ValueError:
        # Fallback to rank-based qcut if duplicate values prevent clean bin edges
        return pd.qcut(series.rank(method='first'), q=5, labels=labels)

df['Median House Price Quintile'] = calculate_quintile_with_labels(df['Median House Price 2022'], decimals=0)
df['Total Crime Quintile'] = calculate_quintile_with_labels(df['Total Crime 2022'], decimals=0)
df['Violence Against The Person Quintile'] = calculate_quintile_with_labels(df['Violence Against The Person'], decimals=0)
df['Total IMD Quintile'] = calculate_quintile_with_labels(df['IMD Score'], decimals=2)

# Clean PTAL for transport
df['Transport PTAL'] = df['PTAL'].astype(str).str.replace('a', '', case=False).str.replace('b', '', case=False).str.strip()

print("Quintiles and PTAL successfully calculated with range labels.")

In [4]:
# 3. Select columns and save to CSV
output_cols = [
    'LSOA code', 
    'LSOA name', 
    'Local authority name',
    'Median House Price Quintile',
    'Total Crime Quintile',
    'Violence Against The Person Quintile',
    'Total IMD Quintile',
    'Transport PTAL'
]

quintiles_df = df[output_cols]
quintiles_df.to_csv('quintiles.csv', index=False)
print("Saved quintiles to quintiles.csv")
quintiles_df.head()